In [12]:
from sklearn.datasets import load_diabetes
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [4]:
# Load the dataset
data = load_diabetes()

X = data.data
y = data.target

print(X.shape)
print(y.shape)
print(data.feature_names)

(442, 10)
(442,)
['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']


In [7]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (353, 10)
Testing : (89, 10)


In [9]:
# Create the XGBoost regression model
model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

In [10]:
# Train the model
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [11]:
# Make the predictions
y_pred = model.predict(X_test)

In [14]:
# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 45.19428866097097
MSE : 3089.2829964747802
RMSE: 55.58131877236074
R²  : 0.4169130777842045


In [21]:
# Let's inspect actual vs predicted
import numpy as np
import pandas as pd

results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

print(results.head(10))

   Actual   Predicted
0   219.0  151.727936
1    70.0  175.137909
2   202.0  151.720398
3   230.0  262.322174
4   111.0  116.184456
5    84.0   96.299568
6   242.0  238.976807
7   272.0  195.626480
8    94.0  147.838135
9    96.0  145.102875


In [16]:
#  Let's predict ONE new patient
sample = X_test[0].reshape(1, -1)

prediction = model.predict(sample)

print("Predicted value:", prediction[0])
print("Actual value   :", y_test[0])

Predicted value: 174.06256
Actual value   : 219.0


## but the scores aren't impressive , Let's use hyper parameter tuning and find the best set

In [17]:
from sklearn.model_selection import RandomizedSearchCV
xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4, 5],
    "subsample": [0.7, 0.8, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=True,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints...
                                          max_leaves=None,
                                          min_child_weight=None, missing=nan,
                                          monotone_constraints=None,
                                          multi_strategy=None,
                                          n_estimators=None, n_jobs=None,
                                          num_parallel_tree=None, ...),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'learning_rate': [0.01, 0.05, 0.1],
                                        'max_depth': [2, 3, 4, 5],
                                        'n_estimators': [100, 200, 300, 500],
                                        'subsample': [0.7, 0.8, 1.0]},
                   random_state=42, scoring='neg_root_mean_squared_error')

In [31]:
# Let's find the best parameters
print('Best parameters:')
print(random_search.best_params_)


Best parameters:
{'subsample': 0.7, 'n_estimators': 500, 'max_depth': 2, 'learning_rate': 0.01}


In [19]:
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)

In [28]:
# Evaluate the best model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :", np.round(mae,2))
print("MSE :", np.round(mse,2))
print("RMSE:", np.round(rmse,2))
print("R²  :", np.round(r2,3))

MAE : 42.12
MSE : 2712.12
RMSE: 52.08
R²  : 0.488


In [29]:
#  Let's predict ONE new patient
sample = X_test[0].reshape(1, -1)

prediction = best_model.predict(sample)

print("Predicted value:", prediction[0])
print("Actual value   :", y_test[0])

Predicted value: 151.72794
Actual value   : 219.0
